## Exercise 3 – Deep Q-Learning (DQN) for CartPole

A system must balance a pole on a moving cart. The goal is to keep the pole upright as long as possible.

### Problem Formulation

| Component | Description |
|-----------|-------------|
| **State** | 4-dimensional continuous vector: `[cart_position, cart_velocity, pole_angle, pole_angular_velocity]` |
| **Actions** | 2 discrete actions – push cart **Left** (0) or **Right** (1) |
| **Reward** | +1 for every timestep the pole stays upright |
| **Episode end** | Pole angle > ±12°, cart moves > ±2.4 units, or 500 timesteps reached |

### Deep Q-Network (DQN) Overview

DQN replaces the traditional Q-table with a neural network `Q(s, a; θ)` and introduces two key stabilisation tricks:

1. **Experience Replay** – store transitions `(s, a, r, s', done)` in a replay buffer and sample mini-batches for training, breaking temporal correlations.
2. **Target Network** – a periodically-copied copy of the main network used to compute TD targets, preventing oscillating updates.

The loss minimised at each training step is:

$$\mathcal{L}(\theta) = \mathbb{E}\left[\left(r + \gamma \max_{a'} Q(s', a'; \theta^{-}) - Q(s, a; \theta)\right)^2\right]$$

where $\theta^{-}$ are the target network weights.

In [ ]:
# Install dependencies if not already available
# !pip install torch gymnasium[classic-control] matplotlib numpy

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim

import gymnasium as gym

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### 1. Q-Network Architecture

A fully-connected neural network that maps a state vector to Q-values for each action.

In [ ]:
class QNetwork(nn.Module):
    """Fully-connected neural network approximating Q(s, a)."""

    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 128):
        super(QNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

### 2. Experience Replay Buffer

In [ ]:
class ReplayBuffer:
    """Fixed-size circular buffer storing environment transitions."""

    def __init__(self, capacity: int = 10_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)

### 3. DQN Agent

In [ ]:
class DQNAgent:
    """
    Deep Q-Network agent with:
    - Epsilon-greedy exploration
    - Experience replay
    - Separate target network updated every `target_update` steps
    """

    def __init__(
        self,
        state_dim: int,
        action_dim: int,
        lr: float = 1e-3,
        gamma: float = 0.99,
        epsilon_start: float = 1.0,
        epsilon_end: float = 0.01,
        epsilon_decay: float = 0.995,
        batch_size: int = 64,
        buffer_capacity: int = 10_000,
        target_update: int = 10,
    ):
        self.action_dim = action_dim
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update = target_update
        self.update_count = 0

        # Online network (trained every step)
        self.q_net = QNetwork(state_dim, action_dim).to(device)
        # Target network (updated periodically)
        self.target_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()
        self.replay_buffer = ReplayBuffer(buffer_capacity)

    # ------------------------------------------------------------------
    # Action selection
    # ------------------------------------------------------------------

    def select_action(self, state: np.ndarray) -> int:
        """ε-greedy action selection."""
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            q_values = self.q_net(state_t)
        return int(q_values.argmax(dim=1).item())

    # ------------------------------------------------------------------
    # Learning step
    # ------------------------------------------------------------------

    def learn(self) -> float:
        """Sample a mini-batch and update the online network."""
        if len(self.replay_buffer) < self.batch_size:
            return 0.0

        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)

        states_t      = torch.tensor(states,      dtype=torch.float32, device=device)
        actions_t     = torch.tensor(actions,     dtype=torch.long,    device=device).unsqueeze(1)
        rewards_t     = torch.tensor(rewards,     dtype=torch.float32, device=device).unsqueeze(1)
        next_states_t = torch.tensor(next_states, dtype=torch.float32, device=device)
        dones_t       = torch.tensor(dones,       dtype=torch.float32, device=device).unsqueeze(1)

        # Current Q-values for taken actions
        current_q = self.q_net(states_t).gather(1, actions_t)

        # Target Q-values (no gradient through target network)
        with torch.no_grad():
            max_next_q = self.target_net(next_states_t).max(dim=1, keepdim=True)[0]
            target_q = rewards_t + self.gamma * max_next_q * (1 - dones_t)

        loss = self.loss_fn(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        # Gradient clipping for stability
        nn.utils.clip_grad_norm_(self.q_net.parameters(), max_norm=10.0)
        self.optimizer.step()

        return loss.item()

    # ------------------------------------------------------------------
    # Target network update
    # ------------------------------------------------------------------

    def update_target_network(self):
        """Hard copy of online weights into target network."""
        self.target_net.load_state_dict(self.q_net.state_dict())

    # ------------------------------------------------------------------
    # Epsilon decay
    # ------------------------------------------------------------------

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

### 4. Training Loop

In [ ]:
def train_dqn(num_episodes: int = 500, print_every: int = 50) -> dict:
    """
    Train a DQN agent on CartPole-v1.

    Returns a dict with episode rewards, losses, and epsilon history.
    """
    env = gym.make('CartPole-v1')
    env.reset(seed=SEED)

    state_dim  = env.observation_space.shape[0]  # 4
    action_dim = env.action_space.n               # 2

    agent = DQNAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        lr=1e-3,
        gamma=0.99,
        epsilon_start=1.0,
        epsilon_end=0.01,
        epsilon_decay=0.995,
        batch_size=64,
        buffer_capacity=10_000,
        target_update=10,
    )

    episode_rewards = []
    losses = []
    epsilons = []

    for episode in range(1, num_episodes + 1):
        state, _ = env.reset()
        total_reward = 0
        episode_loss = []

        while True:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.replay_buffer.push(state, action, reward, next_state, float(done))
            state = next_state
            total_reward += reward

            loss = agent.learn()
            if loss:
                episode_loss.append(loss)

            if done:
                break

        # Decay epsilon after each episode
        agent.decay_epsilon()

        # Sync target network every `target_update` episodes
        if episode % agent.target_update == 0:
            agent.update_target_network()

        episode_rewards.append(total_reward)
        losses.append(np.mean(episode_loss) if episode_loss else 0.0)
        epsilons.append(agent.epsilon)

        if episode % print_every == 0:
            avg_reward = np.mean(episode_rewards[-print_every:])
            print(
                f'Episode {episode:4d}/{num_episodes} | '
                f'Avg Reward (last {print_every}): {avg_reward:6.1f} | '
                f'Epsilon: {agent.epsilon:.3f}'
            )

    env.close()
    return {'rewards': episode_rewards, 'losses': losses, 'epsilons': epsilons}


history = train_dqn(num_episodes=500, print_every=50)

### 5. Results – Episode Reward vs Episodes

In [ ]:
def moving_average(data, window=20):
    return np.convolve(data, np.ones(window) / window, mode='valid')


rewards  = history['rewards']
losses   = history['losses']
epsilons = history['epsilons']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Episode reward ---
ax = axes[0]
ax.plot(rewards, alpha=0.4, color='steelblue', label='Raw')
ax.plot(
    range(19, len(rewards)),
    moving_average(rewards, 20),
    color='steelblue', linewidth=2, label='Moving avg (20)'
)
ax.axhline(y=475, color='red', linestyle='--', linewidth=1, label='Solved (475)')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('CartPole-v1 – Episode Reward (DQN)')
ax.legend()

# --- Training loss ---
ax = axes[1]
ax.plot(losses, color='coral', alpha=0.7)
ax.set_xlabel('Episode')
ax.set_ylabel('MSE Loss')
ax.set_title('Average Training Loss per Episode')

# --- Epsilon schedule ---
ax = axes[2]
ax.plot(epsilons, color='forestgreen')
ax.set_xlabel('Episode')
ax.set_ylabel('Epsilon')
ax.set_title('Exploration Rate (ε) Schedule')

plt.tight_layout()
plt.show()

print(f'\nMax reward achieved : {max(rewards):.0f}')
print(f'Final avg reward (last 50 episodes): {np.mean(rewards[-50:]):.1f}')

### 6. Evaluation – Greedy Policy

Run the trained agent with ε = 0 (pure exploitation) for 10 episodes.

In [ ]:
def evaluate(agent: DQNAgent, num_episodes: int = 10):
    env = gym.make('CartPole-v1')
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0  # Pure exploitation

    eval_rewards = []
    for ep in range(1, num_episodes + 1):
        state, _ = env.reset()
        total_reward = 0
        done = False
        while not done:
            action = agent.select_action(state)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
        eval_rewards.append(total_reward)
        print(f'  Eval episode {ep:2d}: reward = {total_reward:.0f}')

    env.close()
    agent.epsilon = original_epsilon
    print(f'\nMean eval reward: {np.mean(eval_rewards):.1f}')
    return eval_rewards


# Re-create agent from saved weights is not needed here since the
# `agent` variable is still in scope from train_dqn (Python global).
# For a clean evaluation we rebuild agent with the trained weights.

eval_env = gym.make('CartPole-v1')
state_dim  = eval_env.observation_space.shape[0]
action_dim = eval_env.action_space.n
eval_env.close()

trained_agent = DQNAgent(state_dim, action_dim)

# Reload the weights from the last training run
# (we need to re-run train_dqn returning the agent, or just reuse history)
# For convenience we train a small wrapper that also returns the agent:

def train_dqn_with_agent(num_episodes=500, print_every=50):
    env = gym.make('CartPole-v1')
    env.reset(seed=SEED)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.n
    agent = DQNAgent(
        state_dim=state_dim, action_dim=action_dim,
        lr=1e-3, gamma=0.99,
        epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
        batch_size=64, buffer_capacity=10_000, target_update=10,
    )
    episode_rewards, losses, epsilons = [], [], []
    for episode in range(1, num_episodes + 1):
        state, _ = env.reset()
        total_reward = 0
        episode_loss = []
        while True:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            agent.replay_buffer.push(state, action, reward, next_state, float(done))
            state = next_state
            total_reward += reward
            loss = agent.learn()
            if loss:
                episode_loss.append(loss)
            if done:
                break
        agent.decay_epsilon()
        if episode % agent.target_update == 0:
            agent.update_target_network()
        episode_rewards.append(total_reward)
        losses.append(np.mean(episode_loss) if episode_loss else 0.0)
        epsilons.append(agent.epsilon)
        if episode % print_every == 0:
            avg = np.mean(episode_rewards[-print_every:])
            print(f'Episode {episode:4d}/{num_episodes} | Avg Reward: {avg:6.1f} | ε: {agent.epsilon:.3f}')
    env.close()
    return agent, {'rewards': episode_rewards, 'losses': losses, 'epsilons': epsilons}


print('Training agent (this may take a minute)...')
trained_agent, history2 = train_dqn_with_agent(num_episodes=500, print_every=50)
print('\n--- Evaluation (greedy policy) ---')
eval_rewards = evaluate(trained_agent, num_episodes=10)

### Summary

| Aspect | Detail |
|--------|--------|
| **Environment** | `CartPole-v1` (OpenAI Gymnasium) |
| **Algorithm** | Deep Q-Network (DQN) – Mnih et al. 2015 |
| **Network** | 2-layer MLP (128 hidden units, ReLU) |
| **Replay buffer** | 10 000 transitions, mini-batch of 64 |
| **Target update** | Hard copy every 10 episodes |
| **Exploration** | ε-greedy decaying from 1.0 → 0.01 (×0.995/episode) |
| **Optimiser** | Adam, lr = 1e-3 |
| **Discount (γ)** | 0.99 |
| **Solved threshold** | Mean reward ≥ 475 over 100 consecutive episodes |